# Preprocessing Pipeline — Coffee Bean Quality Detection
### Menjalankan `scripts/preprocess_dataset.py` di Kaggle, lalu push hasilnya ke R2 via DVC

Notebook ini adalah **runner**, bukan notebook eksplorasi — logika preprocessing-nya
sendiri hidup di `scripts/preprocess_dataset.py` (bagian dari repo, sudah divalidasi
lokal) supaya reproducible lewat `dvc repro`, bukan cuma kode yang tertanam di sel
notebook. Rujukan lengkap tiap keputusan preprocessing ada di
[`docs/preprocessing-recommendations.md`](https://github.com/Ardiyanto24/coffee-bean-quality-detection/blob/main/docs/preprocessing-recommendations.md).

**Catatan penting soal split** — folder `dataset/test/` yang ada sekarang berisi data
**real-world tanpa label** (bukan test set evaluasi yang sah — labelnya memang tidak
pernah diketahui). Karena itu, notebook ini membuat test split LABELED yang baru
dengan cara mengambil satu fold cluster-aware dari pool train berlabel, lalu
me-rename folder lama itu jadi `real_world/` supaya tidak ada lagi yang keliru
menganggapnya sebagai test set.

**Yang dilakukan pipeline ini** (lihat docstring `scripts/preprocess_dataset.py` untuk detail):
1. Exclude exact-duplicate redundant + cross-class near-duplicate dari pool label.
2. Cluster near-duplicate same-class yang tersisa (union-find) supaya satu bean fisik
   tidak pernah terbelah antara train dan held-out test.
3. `StratifiedGroupKFold(n_splits=5)` — 1 fold jadi held-out **labeled test** baru,
   4 fold sisanya jadi training pool (bisa dipakai 4-fold CV).
4. Crop ke bounding-box foreground (+margin) lalu resize ke 224×224 untuk SEMUA
   gambar yang dipertahankan (train baru, test baru, dan real_world).
5. Simpan `dataset_preprocessed/` + `metadata/manifest_preprocessed.csv`, lalu
   `dvc push` ke R2.

**Prasyarat Kaggle:** sama seperti notebook EDA sebelumnya — Internet On,
`R2_ACCESS_KEY_ID`/`R2_SECRET_ACCESS_KEY` tersedia lewat private dataset
`r2-credentials` (kredensial yang sama juga dipakai untuk `dvc push`, jadi pastikan
access key-nya punya izin tulis ke bucket R2). Tidak butuh GPU — murni image
processing CPU.

In [ ]:
# Sub-Step 1
# Tujuan: Install dependency & clone kode terbaru dari GitHub

!pip install -q "dvc[s3]"

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

# Lokasi mount dataset di /kaggle/input pernah berubah antar kernel (kadang
# /kaggle/input/<slug>/, kadang /kaggle/input/datasets/<slug>/) -- cari filenya
# lewat rglob supaya tidak bergantung pada satu struktur path yang diasumsikan.
matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = matches[0] if matches else None
print("Kandidat ditemukan:", matches)
if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 3
# Tujuan: Tarik dataset mentah + manifest dari R2 (dvc pull)

!dvc pull -v
from pathlib import Path
if not Path("metadata/manifest.csv").exists():
    os.system("python scripts/generate_manifest.py")
print("Manifest tersedia:", Path("metadata/manifest.csv").exists())
print("Dataset train:", len(list(Path("dataset/train").rglob("*.jpg"))), "gambar")
print("Dataset test (real-world, unlabeled):", len(list(Path("dataset/test").glob("*.jpg"))), "gambar")

## Menjalankan Pipeline Preprocessing

`dvc repro preprocess_dataset` menjalankan `scripts/preprocess_dataset.py` HANYA jika
dependensinya (script, `dataset/`, `metadata/manifest.csv`) berubah dibanding
`dvc.lock` — idiomatik DVC, sama seperti stage `generate_manifest` yang sudah ada.

In [ ]:
# Sub-Step 4
# Tujuan: Jalankan stage preprocess_dataset lewat dvc repro

!dvc repro preprocess_dataset -v

In [ ]:
# Sub-Step 5
# Tujuan: Verifikasi & ringkas hasil preprocessing

import pandas as pd

out_manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
print("Total gambar hasil preprocessing:", len(out_manifest))
print()
print(out_manifest["split"].value_counts())
print()
print("Distribusi kelas per split (train & test):")
print(out_manifest[out_manifest["split"] != "real_world"].groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
# Sub-Step 6
# Tujuan: Sanity check visual: satu contoh per split, sebelum vs sesudah crop+resize

import matplotlib.pyplot as plt
from PIL import Image

samples = out_manifest.groupby("split").first().reset_index()
fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
for ax, (_, row) in zip(axes, samples.iterrows()):
    img = Image.open(Path("dataset_preprocessed") / row["image_path"])
    ax.imshow(img)
    ax.set_title(f"{row['split']}\n{img.size[0]}x{img.size[1]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Push Hasil Preprocessing ke R2

`dvc push` mengunggah cache stage `preprocess_dataset` (`dataset_preprocessed/` +
`metadata/manifest_preprocessed.csv`) ke remote R2 yang sama dengan dataset mentah —
kredensial yang sama dari Section 2 harus punya izin tulis ke bucket.

In [ ]:
# Sub-Step 7
# Tujuan: Push dataset_preprocessed ke R2

!dvc push -v

In [ ]:
# Sub-Step 8
# Tujuan: Cetak dvc.lock final untuk disalin kembali ke repo lokal (commit metadata, bukan datanya)

print(Path("dvc.lock").read_text())

## Langkah Selanjutnya (di luar notebook ini)

1. Salin isi `dvc.lock` dari output sel terakhir ke atas ke file `dvc.lock` di repo lokal,
   lalu `git add dvc.lock && git commit && git push` — ini yang menautkan histori git
   ke versi data yang baru saja di-push ke R2 (mengikuti pola yang sama seperti
   `dataset.dvc` untuk data mentah).
2. Siapa pun (termasuk kernel training berikutnya) yang perlu dataset hasil
   preprocessing ini tinggal `dvc pull` seperti biasa setelah `dvc.lock` ter-commit.
3. `dataset_preprocessed/real_world/` **tidak punya label** — jangan pernah dipakai
   untuk menghitung metrik evaluasi (akurasi/F1/dsb). Gunakan hanya `train/` dan
   `test/` untuk training & evaluasi model; `real_world/` untuk sanity-check
   inference akhir saja.